<a href="https://colab.research.google.com/github/202511040-dev/Deep_Learnign_Project/blob/main/DL_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📄 Adaptive PDF Summarization System using Deep Learning

##  Project Overview
This project aims to develop an end-to-end deep learning system capable of generating concise and meaningful summaries from long PDF documents. The system addresses the challenge of information overload by automatically extracting and condensing key insights from large textual data.

---

##  Objectives
- Develop a transformer-based summarization system using a pretrained model (BART)
- Fine-tune the model for abstractive text summarization
- Handle long PDF documents beyond model token limits
- Provide user-controlled summary lengths (short, medium, detailed)
- Build an end-to-end pipeline for PDF upload and summary generation

---

## Key Concepts Used
- Transformer Architecture (Encoder-Decoder)
- Abstractive Text Summarization
- Transfer Learning & Fine-Tuning
- Tokenization & Sequence Modeling
- Hierarchical Summarization

---

##  System Pipeline

1. PDF Upload  
2. Text Extraction  
3. Adaptive Chunking (structure-aware splitting)  
4. Chunk-wise Summarization using BART  
5. Hierarchical Summarization (combine + re-summarize)  
6. Final Summary Output (user-controlled length)

---

##  Model Details
- Model: **BART (Bidirectional and Auto-Regressive Transformer)**
- Type: Encoder-Decoder Transformer
- Capability: Handles both understanding and generation of text
- Limitation: Maximum input length (~1024 tokens)

---

##  Key Innovations in This Project
- Adaptive chunking based on document structure (not fixed size)
- Hierarchical summarization for long documents
- User-controlled summary length
- End-to-end system integration (from PDF to summary)

---

##  Tools & Technologies
- Python
- PyTorch
- Hugging Face Transformers
- Datasets Library
- PDF Processing (pdfplumber)
- Google Colab (GPU-enabled environment)

---

##  Problem Statement
Traditional summarization models are designed for short, clean text and fail to handle long, structured PDF documents. This project extends transformer-based models to work efficiently on large documents using intelligent chunking and hierarchical summarization.

---

##  Expected Outcome
- Accurate summaries of long PDF documents
- Reduced reading time and improved information accessibility
- Flexible summarization based on user needs

---





In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [2]:
from dataclasses import dataclass

@dataclass
class Config:
    MODEL_NAME: str = "facebook/bart-large-cnn"
    MAX_INPUT_LENGTH: int = 1024
    MAX_SUMMARY_LENGTH: int = 150
    MIN_SUMMARY_LENGTH: int = 40

config = Config()

In [ ]:
import torch
from transformers import BartForConditionalGeneration, BartTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = BartTokenizer.from_pretrained(config.MODEL_NAME)
model = BartForConditionalGeneration.from_pretrained(config.MODEL_NAME).to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [ ]:
def summarize_text(text: str):
    inputs = tokenizer(
        text,
        max_length=config.MAX_INPUT_LENGTH,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=config.MAX_SUMMARY_LENGTH,
            min_length=config.MIN_SUMMARY_LENGTH,
            num_beams=4,
            length_penalty=2.0,
            early_stopping=True
        )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

In [ ]:
text = """
Artificial Intelligence is transforming industries by enabling machines
to learn from data, automate tasks, and provide intelligent insights.
Deep learning models, especially transformers, have revolutionized
natural language processing tasks such as summarization and translation.
"""

summary = summarize_text(text)

print("Original:\n", text)
print("\nSummary:\n", summary)

Original:
 
Artificial Intelligence is transforming industries by enabling machines
to learn from data, automate tasks, and provide intelligent insights.
Deep learning models, especially transformers, have revolutionized
natural language processing tasks such as summarization and translation.


Summary:
 Artificial Intelligence is transforming industries by enabling machines to learn from data, automate tasks, and provide intelligent insights. Deep learning models, especially transformers, have revolutionizednatural language processing tasks such as summarization and translation.


## Dataset Preparation & Fine-Tuning BART for Summarization

In [ ]:
from datasets import load_dataset

dataset = load_dataset("cnn_dailymail", "3.0.0")

# Reduce dataset size (VERY IMPORTANT for Colab)
train_data = dataset["train"].select(range(8000))
val_data = dataset["validation"].select(range(1000))

print(train_data[0])

README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

{'article': 'LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won\'t cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don\'t plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don\'t think I\'ll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his number one movie on the UK box office char

In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration

model_name = "facebook/bart-large-cnn"

tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

In [ ]:
max_input_length = 1024
max_target_length = 150

def preprocess_function(examples):
    inputs = examples["article"]
    targets = examples["highlights"]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        targets,
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_val = val_data.map(preprocess_function, batched=True)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    logging_steps=100,
    fp16=True,
    push_to_hub=False
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator
)

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:449: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,0.513400,0.567512
2,0.305500,0.604798
3,0.207200,0.680137


TrainOutput(global_step=12000, training_loss=0.35917025899887084, metrics={'train_runtime': 8658.5991, 'train_samples_per_second': 2.772, 'train_steps_per_second': 1.386, 'total_flos': 5.2010510450688e+16, 'train_loss': 0.35917025899887084, 'epoch': 3.0})

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from transformers import BartForConditionalGeneration, BartTokenizer
import torch

MODEL_PATH = "/content/drive/MyDrive/DL_PROJECT/pdf_summarizer_model"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = BartTokenizer.from_pretrained(MODEL_PATH)
model = BartForConditionalGeneration.from_pretrained(MODEL_PATH).to(device)

print("✅ Model Loaded Successfully")

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

✅ Model Loaded Successfully


In [ ]:
# import os

# save_path = "/content/drive/MyDrive/DL_PROJECT/pdf_summarizer_model"

# # Create folder if it doesn't exist
# os.makedirs(save_path, exist_ok=True)

In [ ]:
# model.save_pretrained(save_path)
# tokenizer.save_pretrained(save_path)

('/content/drive/MyDrive/DL_PROJECT/pdf_summarizer_model/tokenizer_config.json',
 '/content/drive/MyDrive/DL_PROJECT/pdf_summarizer_model/special_tokens_map.json',
 '/content/drive/MyDrive/DL_PROJECT/pdf_summarizer_model/vocab.json',
 '/content/drive/MyDrive/DL_PROJECT/pdf_summarizer_model/merges.txt',
 '/content/drive/MyDrive/DL_PROJECT/pdf_summarizer_model/added_tokens.json')

In [ ]:
# print(os.listdir(save_path))

['config.json', 'generation_config.json', 'model.safetensors', 'tokenizer_config.json', 'special_tokens_map.json', 'vocab.json', 'merges.txt']


# Baseline wirthout chunking

In [5]:
def baseline_summary(text, tokenizer, model, device):
    """
    Direct summarization (NO chunking)
    Used as baseline for comparison
    """

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=150,
        min_length=40,
        num_beams=4
    )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    return summary

# PDF Processing + Adaptive Chunking

In [6]:
!pip install -q pdfplumber nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 112.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 115.5 MB/s eta 0:00:00


In [7]:
import pdfplumber
import nltk
import torch

nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [8]:
def extract_text_from_pdf(pdf_path):
    text = ""

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"

    return text

In [9]:
def count_tokens(text, tokenizer):
    return len(tokenizer.encode(text))

In [10]:
import re

def clean_text(text):
    text = re.sub(r'\[\d+(,\s*\d+)*\]', '', text)  # remove citations [1,2]
    text = re.sub(r'\(.*?\)', '', text)           # remove brackets (..)
    text = re.sub(r'http\S+', '', text)           # remove URLs
    text = re.sub(r'\s+', ' ', text)              # normalize spaces
    return text.strip()

In [11]:
def split_by_paragraph(text):
    paragraphs = text.split("\n")
    return [p.strip() for p in paragraphs if len(p.strip()) > 50]

In [12]:
def get_dynamic_max_length(text):
    length = len(text.split())

    if length < 2000:
        return 800
    elif length < 5000:
        return 900
    else:
        return 1000

Adaptive chunking

In [13]:
from nltk.tokenize import sent_tokenize

def split_by_paragraph(text):
    paragraphs = text.split("\n")
    return [p.strip() for p in paragraphs if len(p.strip()) > 50]

def get_dynamic_max_length(text):
    length = len(text.split())
    if length < 2000: return 800
    elif length < 5000: return 900
    else: return 1000

def adaptive_chunking(text, tokenizer):
    paragraphs = split_by_paragraph(text)
    max_tokens = get_dynamic_max_length(text)
    
    chunks = []
    current_chunk = []
    current_length = 0

    for para in paragraphs:
        sentences = sent_tokenize(para)
        for sentence in sentences:
            sentence_length = len(tokenizer.encode(sentence, add_special_tokens=False))
            if current_length + sentence_length > max_tokens and current_chunk:
                chunks.append(" ".join(current_chunk))
                overlap_sentences = current_chunk[-2:] if len(current_chunk) >= 2 else current_chunk
                current_chunk = overlap_sentences
                current_length = sum(len(tokenizer.encode(s, add_special_tokens=False)) for s in current_chunk)

            current_chunk.append(sentence)
            current_length += sentence_length

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

In [14]:
from transformers import BartForConditionalGeneration, BartTokenizer

model_path = "/content/drive/MyDrive/DL_PROJECT/pdf_summarizer_model"

tokenizer = BartTokenizer.from_pretrained(model_path)
model = BartForConditionalGeneration.from_pretrained(model_path).to("cuda")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def rank_chunks(chunks):
    if not chunks:
        return []
    vectorizer = TfidfVectorizer(stop_words='english')
    X = vectorizer.fit_transform(chunks)
    scores = np.sum(X.toarray(), axis=1)
    
    normalized_scores = []
    for i, chunk in enumerate(chunks):
        word_count = len(chunk.split())
        normalized_score = scores[i] / (word_count + 1e-5)
        normalized_scores.append(normalized_score)
        
    ranked = [c for _, c in sorted(zip(normalized_scores, chunks), reverse=True)]
    return ranked

In [16]:
def select_chunks(ranked_chunks, mode):
    """
    Select top chunks based on user mode
    """

    if mode == "short":
        return ranked_chunks[:3]
    elif mode == "detailed":
        return ranked_chunks[:12]
    else:  # medium
        return ranked_chunks[:6]

chunk wise summarization

In [17]:
def get_summary_params(mode="medium"):
    if mode == "short":
        return {
            "max_length": 80,
            "min_length": 30,
            "num_beams": 4
        }
    elif mode == "detailed":
        return {
            "max_length": 250,
            "min_length": 100,
            "num_beams": 5
        }
    else:  # medium
        return {
            "max_length": 150,
            "min_length": 60,
            "num_beams": 4
        }

In [18]:
def get_summary_params(mode):
    if mode == "short": return {"max_length": 80, "min_length": 30, "num_beams": 4}
    elif mode == "detailed": return {"max_length": 250, "min_length": 100, "num_beams": 5}
    else: return {"max_length": 150, "min_length": 60, "num_beams": 4}

def summarize_chunks(chunks, mode="medium"):
    summaries = []
    params = get_summary_params(mode)

    for chunk in chunks:
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True, max_length=1024).to(device)

        ids = model.generate(
            inputs["input_ids"],
            max_length=params["max_length"],
            min_length=params["min_length"],
            num_beams=params["num_beams"],
            length_penalty=2.0
        )

        summaries.append(tokenizer.decode(ids[0], skip_special_tokens=True))

    return summaries

# Hierarchical Summarization

In [19]:
def refine_summary(text, tokenizer, model, device, mode):
    if mode == "short": max_len, min_len, beams = 80, 30, 4
    elif mode == "detailed": max_len, min_len, beams = 300, 120, 6
    else: max_len, min_len, beams = 150, 60, 5

    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024).to(device)

    ids = model.generate(
        inputs["input_ids"],
        max_length=max_len,
        min_length=min_len,
        num_beams=beams,
        length_penalty=2.2,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

    return tokenizer.decode(ids[0], skip_special_tokens=True)

In [20]:
def remove_redundancy(text):
    sentences = text.split(".")

    seen = set()
    unique_sentences = []

    for s in sentences:
        s = s.strip()
        if s and s not in seen:
            seen.add(s)
            unique_sentences.append(s)

    return ". ".join(unique_sentences)

In [21]:
def generate_title(text, tokenizer, model, device):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    ids = model.generate(
        inputs["input_ids"],
        max_length=20,
        num_beams=4
    )

    return tokenizer.decode(ids[0], skip_special_tokens=True)

In [22]:
from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def extract_key_points(text, num_points=5):
    sentences = sent_tokenize(text)
    if len(sentences) <= num_points:
        return sentences

    vectorizer = TfidfVectorizer(stop_words='english')
    try:
        X = vectorizer.fit_transform(sentences)
        scores = np.sum(X.toarray(), axis=1)
        ranked = [s for _, s in sorted(zip(scores, sentences), reverse=True)]
        return ranked[:num_points]
    except ValueError:
        return sentences[:num_points]

In [23]:
def hierarchical_summarization(text, mode="medium"):

    # Step 1: Chunking
    chunks = adaptive_chunking(text, tokenizer)
    print(f"Total chunks before ranking: {len(chunks)}")

    # Step 2: Ranking
    ranked_chunks = rank_chunks(chunks)

    # Step 3: Selection
    selected_chunks = select_chunks(ranked_chunks, mode)
    print(f"Selected chunks: {len(selected_chunks)}")

    # Step 4: Chunk Summarization
    chunk_summaries = summarize_chunks(selected_chunks, mode)

    # Step 5: Combine summaries
    combined_summary = " ".join(chunk_summaries)
    combined_summary = remove_redundancy(combined_summary)

    combined_summary = combined_summary.replace("  ", " ")

    # Step 6: Final Refinement (ONLY ONCE)
    final_summary = refine_summary(combined_summary, tokenizer, model, device, mode)
    final_summary = remove_redundancy(final_summary)

    title = generate_title(final_summary, tokenizer, model, device)
    points = extract_key_points(final_summary)

    return {
        "title": title,
        "summary": final_summary,
        "points": points
    }

    return final_summary

In [24]:
mode = input("Choose summary type (short / medium / detailed): ").strip().lower()

if mode not in ["short", "medium", "detailed"]:
    mode = "medium"

Choose summary type (short / medium / detailed): detailed


In [25]:
pdf_path ="/content/drive/MyDrive/SK_Papers/Drive4GPT.pdf"  # upload your PDF


text = extract_text_from_pdf(pdf_path)
text = clean_text(text)

print("🔵 Running BASELINE summarization...\n")

baseline_output = baseline_summary(text, tokenizer, model, device)

print("Baseline Summary:\n")
print(baseline_output)

print("Chunk based  Summary:\n")
final_summary = hierarchical_summarization(text, mode)

print("\nFinal Summary:\n")
print(final_summary)

🔵 Running BASELINE summarization...

Baseline Summary:

DriveGPT4 represents the pioneering effort to leverage large language models (LLMs) in autonomous driving systems.
It can be used to generate natural language based on data from vehicle-mounted sensor data.
Drivers can use this language to make autonomous decisions without the need for human intervention.
DriveG PT4 can predict slow-level vehicle control needs and eliminate discontinuous intermediate steps.
This study seeks to ensure that vehicles can effectively handle diverse sit-extended scenarios.
Chunk based  Summary:

Total chunks before ranking: 15
Selected chunks: 12


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1695: UserWarning: Unfeasible length constraints: `min_length` (56) is larger than the maximum possible length (20). Generation will stop at the defined maximum length. You should decrease the minimum length and/or increase the maximum length.
  warnings.warn(



Final Summary:

{'title': 'DriveGPT4 is an end-to-end autonomous driving system that', 'summary': 'Authors: Autonomous driving systems could benefit from improved text prediction. DriveGPT4 is an end-to-end autonomous driving system that utilizes large, interpretable text models. It can directly predict planned paths tuning dataset, specifically tailored for autonomous driving ap- and/or low-level vehicle controls. Drivers can interact with the system by interacting with videos and texts. The fine-tuning of domain-specific data enables DriveG PT4 to yield close-orevenimproved results. Back to Mail Online home. Back to the page you came from', 'points': ['Authors: Autonomous driving systems could benefit from improved text prediction', 'DriveGPT4 is an end-to-end autonomous driving system that utilizes large, interpretable text models', 'It can directly predict planned paths tuning dataset, specifically tailored for autonomous driving ap- and/or low-level vehicle controls', 'Drivers ca

In [26]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00


In [27]:
import evaluate
!pip install rouge_score

rouge = evaluate.load("rouge")

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=61bdb301de273027e7ae26c072444ff77c5a0bfb10681da51678d8af1af16d94
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [28]:
def evaluate_summaries(reference, baseline, improved):

    results_baseline = rouge.compute(
        predictions=[baseline],
        references=[reference]
    )

    results_improved = rouge.compute(
        predictions=[improved],
        references=[reference]
    )

    print("🔵 BASELINE ROUGE:")
    print(results_baseline)

    print("\n🟢 IMPROVED MODEL ROUGE:")
    print(results_improved)

In [ ]:
reference_summary = "Write or take actual summary here"

evaluate_summaries(
    reference_summary,
    baseline_output,
    final_summary
)

SyntaxError: unterminated string literal (detected at line 1) (3792015400.py, line 1)

In [ ]:
print("\n🔵 BASELINE SUMMARY:\n", baseline_output)
print("\n🟢 IMPROVED SUMMARY:\n", final_summary)

In [ ]:
import numpy as np
from datasets import load_dataset
import evaluate

# Load the test split of CNN/DailyMail (which was used for training)
print("Loading CNN/DailyMail dataset...")
dataset = load_dataset("cnn_dailymail", "3.0.0")
test_data = dataset["test"].select(range(10)) # evaluating on 10 samples to save time

try:
    rouge = evaluate.load("rouge")
except Exception:
    import evaluate
    rouge = evaluate.load("rouge")

baseline_preds = []
hierarchical_preds = []
references = []

print("Running Evaluation on Test Set...")
for i, example in enumerate(test_data):
    article = example["article"]
    reference = example["highlights"]
    references.append(reference)
    
    # 1. Baseline Summary (Direct BART generation without chunking)
    print(f"Processing sample {i+1}/10...")
    baseline_pred = baseline_summary(article, tokenizer, model, device)
    baseline_preds.append(baseline_pred)
    
    # 2. Hierarchical Summary (New Pipeline)
    try:
        hier_pred = hierarchical_summarization(article, mode="medium")
        # Handle dict return from the updated advanced pipeline
        hier_pred_text = hier_pred["summary"] if isinstance(hier_pred, dict) else hier_pred
    except Exception as e:
        hier_pred_text = baseline_pred # fallback if chunking fails on very short text
        
    hierarchical_preds.append(hier_pred_text)

print("\nCalculating ROUGE Scores...")

baseline_results = rouge.compute(predictions=baseline_preds, references=references)
hierarchical_results = rouge.compute(predictions=hierarchical_preds, references=references)

print("\n🔵 BASELINE (Direct Generation) ROUGE:")
print({k: round(v, 4) for k, v in baseline_results.items()})

print("\n🟢 IMPROVED (Hierarchical Pipeline) ROUGE:")
print({k: round(v, 4) for k, v in hierarchical_results.items()})